# Updating Demand Beliefs with Bayesian Methods

**Scenario.** Chapter & Craft, a fictional bookstore and hobby retail chain, is piloting a
Curated Reader's Box subscription. They must place their box-assembly order with suppliers
two weeks in advance. The key unknown is mean weekly demand per store. You'll build the
Normal-Normal conjugate update to revise that belief twice as pilot scan data comes in, and
translate each updated belief into a recommended pre-order quantity.

See `INSTRUCTIONS.md` for the full prompt and `data/README.md` for the dataset citation.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

BENCH_PATH        = "data/hobby_book_industry_benchmarks.csv"
PILOT_BATCH1_PATH = "data/pilot_scan_data_weeks1_4.csv"
PILOT_BATCH2_PATH = "data/pilot_scan_data_weeks5_8.csv"

N_US_HOBBY_BOOK_STORES = 25_000  # approximate count of US sporting goods/hobby/book stores (US Census)
BOX_SHARE              = 0.008   # curated boxes ≈ 0.8% of category spend (illustrative estimate)
AVG_BOX_PRICE          = 24.0    # average Chapter & Craft box price ($)
PRIOR_SD               = 7.0     # units — wider than historical variability; new-product uncertainty
Q_BUFFER               = 0.5     # pre-order buffer: Q = posterior_mean + Q_BUFFER × posterior_sd

## 1. Derive the prior mean from hobby/book sales data

Load the monthly sporting goods/hobby/book store sales data and compute the average
monthly hobby/book store sales for 2022–2024. Derive `prior_mu` — mean weekly box units
per store:

1. Weekly sales per store = (avg monthly $) / `N_US_HOBBY_BOOK_STORES` / 4.33 weeks
2. Weekly box $ per store = weekly sales × `BOX_SHARE`
3. Units = box $ / `AVG_BOX_PRICE`

Then apply an early-stage discount: multiply by 0.75 (new program, narrower reach).

In [ ]:
# TODO: Load the benchmark CSV into a DataFrame called `bench`.
bench = ...

# TODO: Compute `avg_monthly_usd_m` — average monthly hobby/book store sales in $M for 2022-2024.
avg_monthly_usd_m = ...

# TODO: Derive `prior_mu` (see steps above).
prior_mu = ...

print(f"Avg monthly US hobby/book store sales (2022-24): ${avg_monthly_usd_m:,.0f}M")
print(f"Derived prior_mu: {prior_mu:.1f} units/store/week")
print(f"Prior: Demand ~ Normal({prior_mu:.1f}, {PRIOR_SD:.1f}²)")

## 2. Load pilot data: first batch (weeks 1–4)

Load `pilot_scan_data_weeks1_4.csv`. Weeks 5–8 haven't happened yet from the analyst's
point of view — that data lives in a separate file we don't touch until later.

In [ ]:
# TODO: Load PILOT_BATCH1_PATH into `batch1`. Display it.
batch1 = ...

## 3. Compute batch 1's likelihood parameters

The likelihood is summarized by the sample mean and standard error of the weekly
unit counts: `se = std(ddof=1) / sqrt(n_weeks)`.

In [ ]:
# TODO: Compute lik1_mu, lik1_sd (mean and se of mean_units_sold in batch1).
lik1_mu = ...
lik1_sd = ...

print(f"Batch 1 likelihood: mu={lik1_mu:.1f}, se={lik1_sd:.2f}")

## 4. Implement `normal_update`

The precision of a Normal distribution is 1 / sd². The conjugate update combines
the prior and likelihood by adding their precisions and taking a precision-weighted
average of their means:

  prior_prec = 1 / prior_sd²
  lik_prec   = 1 / lik_sd²
  post_prec  = prior_prec + lik_prec          (precisions add)
  post_mu    = (prior_prec × prior_mu + lik_prec × lik_mu) / post_prec
  post_sd    = sqrt(1 / post_prec)

In [ ]:
# TODO: Implement normal_update using the formula above. Return (posterior_mean, posterior_sd).
def normal_update(prior_mu: float, prior_sd: float,
                  lik_mu: float, lik_sd: float) -> tuple[float, float]:
    """Conjugate Normal-Normal update. Returns (posterior_mean, posterior_sd)."""
    ...

## 5. First update: prior + batch 1

In [ ]:
# TODO: Compute post1_mu, post1_sd using normal_update.
post1_mu, post1_sd = ...

print(f"Posterior 1: mu={post1_mu:.1f}, sd={post1_sd:.2f}")
print(f"  Sd reduced from {PRIOR_SD:.1f} → {post1_sd:.2f}")

## 6. Two weeks later: batch 2 arrives (weeks 5–8)

With Posterior 1 in hand, the pilot keeps running. Now load the next four weeks of scan
data from a separate file — the batch that wasn't available yet when Posterior 1 was
computed.

In [ ]:
# TODO: Load PILOT_BATCH2_PATH into `batch2`. Display it.
batch2 = ...

## 7. Compute batch 2's likelihood parameters

In [ ]:
# TODO: Compute lik2_mu, lik2_sd (mean and se of mean_units_sold in batch2).
lik2_mu = ...
lik2_sd = ...

print(f"Batch 2 likelihood: mu={lik2_mu:.1f}, se={lik2_sd:.2f}")

## 8. Second update: Posterior 1 + batch 2

In [ ]:
# TODO: Compute post2_mu, post2_sd using normal_update.
post2_mu, post2_sd = ...

print(f"Posterior 2: mu={post2_mu:.1f}, sd={post2_sd:.2f}")
print(f"  Sd reduced from {post1_sd:.2f} → {post2_sd:.2f}")

## 9. Plot all three belief distributions

Plot the prior, Posterior 1, and Posterior 2 as Normal pdfs on one set of axes, using
`scipy.stats.norm.pdf` for the y-values. Add a legend and labeled axes.

In [ ]:
# TODO: Plot the three distributions.

## 10. Recommended pre-order quantity Q under each belief state

Q = posterior_mean + Q_BUFFER × posterior_sd
Chapter & Craft wants a small buffer above the mean to limit stockouts, but not so large
they risk heavy inventory waste.

In [ ]:
# TODO: Compute Q_prior, Q_post1, Q_post2 and print them.
Q_prior = ...
Q_post1 = ...
Q_post2 = ...

# TODO: Print a summary comparing Q across all three belief states.